In [2]:
import pandas as pd
import json
import re
from openai import AzureOpenAI
from dotenv import load_dotenv
import os

load_dotenv()  # loads the .env file

True

In [3]:
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2025-01-01-preview"
)

In [4]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "Say hello in one word"}],
    max_tokens=10,
)
print(response.choices[0].message.content)

Hello!


In [5]:
df = pd.read_csv("data/Reddit_askdocs_2k.csv")
df["question_text"] = df["title"].fillna("").astype(str) + " " + df["selftext"].fillna("").astype(str)
sample_df = df.sample(n=25, random_state=42).reset_index(drop=True)
print(f"Loaded {len(sample_df)} questions")
sample_df["question_text"].head(3)

Loaded 25 questions


0    I used to be malnourished for 1.5 years as an ...
1    My dad (70M) was informed that he needs to sta...
2    A question about anesthesia  When getting a pl...
Name: question_text, dtype: object

In [6]:
import sys
print(sys.version)  # should show 3.9.x

import scispacy
import spacy
nlp = spacy.load("en_core_sci_sm")
print("scispaCy works in notebook!")

3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 18:02:02) 
[Clang 18.1.8 ]


/opt/anaconda3/envs/scispacy_env/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


scispaCy works in notebook!


/opt/anaconda3/envs/scispacy_env/lib/python3.9/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [7]:
def extract_medical_entities(text):
    doc = nlp(text)
    entities = [ent.text for ent in doc.ents if len(ent.text) > 2]  
    return entities

In [8]:
# Load SciBERT via scispaCy
nlp_scibert = spacy.load("en_core_sci_scibert")

def extract_entities_scibert(text):
    doc = nlp_scibert(text)
    entities = [ent.text for ent in doc.ents if len(ent.text) > 2]
    return entities

# Compare both models on same question
test_text = sample_df["question_text"].iloc[0]
print("scispaCy (en_core_sci_sm):", extract_medical_entities(test_text))
print("\nSciBERT (en_core_sci_scibert):", extract_entities_scibert(test_text))

scispaCy (en_core_sci_sm): ['years', 'early teen', "I'm", 'eating', 'will I permanently stay', 'height', 'male', 'parents', "5'2.5"]

SciBERT (en_core_sci_scibert): ['years', 'early', 'teen', "I'm", 'eating', 'height', 'male', 'parents', "5'2.5"]


/opt/anaconda3/envs/scispacy_env/lib/python3.9/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
/opt/anaconda3/envs/scispacy_env/lib/python3.9/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


In [9]:
# At the bottom of the SciBERT cell, define test_text inline:
test_text = sample_df["question_text"].iloc[0]
print("SciBERT entities:", extract_entities_scibert(test_text))

SciBERT entities: ['years', 'early', 'teen', "I'm", 'eating', 'height', 'male', 'parents', "5'2.5"]


/opt/anaconda3/envs/scispacy_env/lib/python3.9/site-packages/thinc/shims/pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
/opt/anaconda3/envs/scispacy_env/lib/python3.9/site-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


In [10]:
test_text = sample_df["question_text"].iloc[0]
print("Question:", test_text[:300])
print("\nExtracted entities:", extract_medical_entities(test_text))

Question: I used to be malnourished for 1.5 years as an early teen. Now that I'm eating well, will I grow some more or will I permanently stay at this height? I'm a male who is 16 and 5'7", and my parents are 5'9.5" and 5'2.5".

Extracted entities: ['years', 'early teen', "I'm", 'eating', 'will I permanently stay', 'height', 'male', 'parents', "5'2.5"]


In [11]:
def find_medical_relationships(question_text, entities):
    if len(entities) < 2:
        return []
    
    prompt = f"""You are a strict medical NLP assistant. Your job is to find ONLY real, specific medical relationship pairs from a list of terms.

ALLOWED relationship types (use exactly these labels):
- "drug-disease": a specific drug/medication treats a specific disease
- "disease-symptom": a specific disease causes a specific symptom
- "symptom-disease": a specific symptom indicates a specific disease

STRICT RULES:
1. Both terms MUST be real, specific medical entities:
   - VALID: disease names, symptom names, drug names
   - INVALID: test results (positive, negative, normal), procedures (ultrasound, MRI), vague words (issues, problems, positive, normal, stuff), body parts alone (kidney, chest), emotions alone (scared, worried)
2. The two terms must NOT be the same thing worded differently (e.g. "allergy" and "allergies", "rash" and "heat rash")
3. Do NOT create reverse pairs — if you write (A, B), do NOT also write (B, A)
4. If fewer than 2 valid medical entities exist, return empty list
5. Only pair things with a DIRECT medical relationship — not just because they appear together

GOOD examples:
- ["diabetes", "kidney failure", "disease-symptom"] ✓
- ["metformin", "diabetes", "drug-disease"] ✓
- ["chest pain", "heart attack", "symptom-disease"] ✓

BAD examples (do not do these):
- ["covid", "positive", "disease-symptom"] ✗ (positive is not a symptom)
- ["allergy", "allergies", "disease-symptom"] ✗ (same thing)
- ["rash", "rash", "disease-symptom"] ✗ (same thing)
- ["flu", "runny nose", "disease-symptom"] AND ["runny nose", "flu", "symptom-disease"] ✗ (reverse duplicate)

Terms extracted: {entities}

Original question for context: \"\"\"{question_text[:400]}\"\"\"

Return ONLY valid JSON, no markdown:
{{"pairs": [["term1", "term2", "relationship_type"], ...]}}
"""
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=400,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"^```(?:json)?", "", raw).strip()
    raw = re.sub(r"```$", "", raw).strip()
    result = json.loads(raw)
    return result.get("pairs", [])

In [17]:
import time

results = []

for i, row in sample_df.iterrows():
    text = row["question_text"]
    entities = extract_medical_entities(text)
    try:
        pairs = find_medical_relationships(text, entities)
    except Exception as e:
        pairs = []
        print(f"Row {i} error: {e}")
    
    results.append({
        "question_text": text,
        "entities": entities,
        "relationship_pairs": pairs
    })
    
    time.sleep(0.5)
    print(f"✓ {i+1}/10")

results_df = pd.DataFrame(results)
print("\nDone!")

✓ 1/10
✓ 2/10
✓ 3/10
✓ 4/10
✓ 5/10
✓ 6/10
✓ 7/10
✓ 8/10
✓ 9/10
✓ 10/10
✓ 11/10
✓ 12/10
✓ 13/10
✓ 14/10
✓ 15/10
✓ 16/10
✓ 17/10
✓ 18/10
✓ 19/10
✓ 20/10
✓ 21/10
✓ 22/10
✓ 23/10
✓ 24/10
✓ 25/10

Done!


In [19]:
for i, row in results_df.iterrows():
    print(f"\n{'='*60}")
    print(f"Question {i+1}: {row['question_text'][:150]}")
    print(f"Pairs: {row['relationship_pairs']}")


Question 1: I used to be malnourished for 1.5 years as an early teen. Now that I'm eating well, will I grow some more or will I permanently stay at this height? I
Pairs: []

Question 2: My dad (70M) was informed that he needs to start dialysis. Anything we can do? My dad who is (6feet tall and about 260lbs) came home crying that his d
Pairs: [['diabetes', 'dialysis', 'disease-symptom']]

Question 3: A question about anesthesia  When getting a planned procedure, you're told to fast the night before as to not risk aspiration. Done that twice, once w
Pairs: []

Question 4: Hi! I (26M) cut my finger and lost feeling in the tip, what are the chances of the nerve repairing itself? 
My finger got hit by a crazy sharp knife t
Pairs: []

Question 5: Is it crazy to stay in a hotel to avoid family members with covid? I m27 am vaccinated. Both of my parents who I live with (also vaccinated) have covi
Pairs: [['covid', 'isolating', 'disease-symptom']]

Question 6: pulsating in my right collarbone,

In [12]:
import time

# Judge client — same endpoint, different deployment
judge_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2025-01-01-preview"
)

def judge_pairs(question, pairs):
    if not pairs:
        return []
    
    prompt = f"""You are a medical expert reviewing relationship pairs extracted from a patient question.

Question: {question}

Extracted pairs: {pairs}

For each pair, respond with:
- VALID or INVALID
- One sentence reason

Format: (entity1, entity2, type) → VALID/INVALID: reason

Only respond with the judgments, nothing else."""

    response = judge_client.chat.completions.create(
        model="gpt-4o",   # your judge deployment
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

# Test on first question
test_q = sample_df["question_text"].iloc[0]
test_pairs = [("malnourishment", "height", "disease-symptom")]  # example pair
print(judge_pairs(test_q, test_pairs))

('malnourishment', 'height', 'disease-symptom') → INVALID: malnourishment is a condition, not a symptom, and height is not a disease.


In [30]:
# Run judge on all questions
judge_results = []

for i, row in results_df.iterrows():
    question = row["question_text"]
    pairs = row["relationship_pairs"]
    
    if pairs:
        judgment = judge_pairs(question, pairs)
    else:
        judgment = "No pairs to judge"
    
    judge_results.append(judgment)
    print(f"Q{i+1} done")
    time.sleep(1)  # avoid rate limits

results_df["judge_feedback"] = judge_results
print("\nDone!")

Q1 done
Q2 done
Q3 done
Q4 done
Q5 done
Q6 done
Q7 done
Q8 done
Q9 done
Q10 done
Q11 done
Q12 done
Q13 done
Q14 done
Q15 done
Q16 done
Q17 done
Q18 done
Q19 done
Q20 done
Q21 done
Q22 done
Q23 done
Q24 done
Q25 done

Done!


In [31]:
for i, row in results_df.iterrows():
    print(f"\n{'='*60}")
    print(f"Q{i+1}: {row['question_text'][:150]}")
    
    feedback = row["judge_feedback"]
    
    if feedback == "No pairs to judge":
        print("No pairs to judge")
        continue
    
    # All pairs
    print(f"\nAll Pairs: {row['relationship_pairs']}")
    
    # Judgments with valid/invalid labels
    lines = [line.strip() for line in feedback.split("\n") if line.strip()]
    print("\nJudgments:")
    for line in lines:
        if "INVALID" in line:
            print(f"  ❌ {line}")
        elif "VALID" in line:
            print(f"  ✅ {line}")
    
    # Valid pairs only
    valid_pairs = [line.strip() for line in lines if "VALID" in line and "INVALID" not in line]
    print(f"\nValid Pairs Only ({len(valid_pairs)}):")
    if valid_pairs:
        for v in valid_pairs:
            print(f"  → {v}")
    else:
        print("  None")


Q1: I used to be malnourished for 1.5 years as an early teen. Now that I'm eating well, will I grow some more or will I permanently stay at this height? I
No pairs to judge

Q2: My dad (70M) was informed that he needs to start dialysis. Anything we can do? My dad who is (6feet tall and about 260lbs) came home crying that his d

All Pairs: [['diabetes', 'dialysis', 'disease-symptom']]

Judgments:
  ❌ (diabetes, dialysis, disease-symptom) → INVALID: Diabetes is a risk factor for kidney disease but not a direct symptom of dialysis.

Valid Pairs Only (0):
  None

Q3: A question about anesthesia  When getting a planned procedure, you're told to fast the night before as to not risk aspiration. Done that twice, once w
No pairs to judge

Q4: Hi! I (26M) cut my finger and lost feeling in the tip, what are the chances of the nerve repairing itself? 
My finger got hit by a crazy sharp knife t
No pairs to judge

Q5: Is it crazy to stay in a hotel to avoid family members with covid? I m27 am vacci